# 🔬 MILK10k Skin Lesion Classification: Gradual Unfreezing ResNet50
### A 4-Stage Progressive Fine-Tuning Benchmark

---

## 📌 Experiment Overview
In this experiment, we compare our previous standard ImageNet **Base ResNet50** against a **Gradual Unfreezing** strategy.
Instead of unfreezing all layers at once (which can distort pretrained ImageNet feature representations), we progressively thaw layers:

$$\text{Stage 1: Head Only (FC)} \longrightarrow \text{Stage 2: + Layer 4} \longrightarrow \text{Stage 3: + Layer 3} \longrightarrow \text{Stage 4: Full Network}$$

### 🎯 Key Scientific Safeguards:
- **Identical Split**: Uses the exact same lesion-level stratified 70/15/15 split (Seed 42) from the baseline to ensure strict, fair scientific comparison.
- **Dermoscopic Only**: Same $224 \times 224$ dermoscopic images with identical standard preprocessing.
- **Primary Selection Metric**: Validation **Macro F1** (preserving rare classes like `MAL_OTH`, `VASC`).
- **Comparative Benchmark**: Evaluates best checkpoint on the test set and directly compares results with Base ResNet50.

## 🛠️ Cell 0: GPU Verification & Environment Setup

In [ ]:
!nvidia-smi
!pip install -q --upgrade pip
!pip install -q kaggle scikit-learn seaborn matplotlib tqdm pillow torch torchvision

## 📥 Cell 1: Dataset Verification / Download
Verifies that the MILK10k dataset is present in `/content/dataset_milk10k`, or downloads it via Kaggle.

In [ ]:
import os
from pathlib import Path

DATA_DIR = "/content/dataset_milk10k"
KAGGLE_DATASET = "nguyenphucduyloc/milk10k-isic-challenge-2025"

existing_csvs = list(Path(DATA_DIR).rglob("*GroundTruth*.csv")) if Path(DATA_DIR).exists() else []
if existing_csvs:
    print(f"✓ Dataset already present in '{DATA_DIR}'. Skipping download.")
else:
    print("Dataset not found locally. Setting up Kaggle download...")
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME') or os.environ.get('KAGGLE_USERNAME', '')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY') or os.environ.get('KAGGLE_KEY', '')
    except Exception:
        pass

    if not os.environ.get('KAGGLE_USERNAME') or not os.environ.get('KAGGLE_KEY'):
        os.environ['KAGGLE_USERNAME'] = 'tamal247'
        os.environ['KAGGLE_KEY']      = 'c8446b8c8bd05c1d059d7bab184de6d2'

    os.makedirs(DATA_DIR, exist_ok=True)
    !kaggle datasets download -d {KAGGLE_DATASET} --unzip -p {DATA_DIR}
    print("✓ Dataset ready in:", DATA_DIR)

## ⚙️ Cell 2: Configuration & 4-Stage Parameters

In [ ]:
import time, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import ResNet50_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, classification_report, confusion_matrix
)

# ------------------------------------------------------------------------------
# 4-STAGE GRADUAL UNFREEZING CONFIG
# ------------------------------------------------------------------------------
CONFIG = {
    "experiment_name": "ResNet50 Gradual Unfreezing (4 Stages)",
    "data_dir": "/content/dataset_milk10k",
    "output_dir": "outputs/gradual_unfreezing_resnet50",
    "base_model_results_dir": "outputs/base_resnet50/results",
    "split_seed": 42,
    "seed": 42,
    "batch_size": 32,
    "num_workers": 2,
    "use_amp": True,
    "patience": 7,
    # Stage Setup (Epochs & Learning Rates)
    "stages": [
        {"stage": 1, "name": "Head Only (FC)",       "epochs": 5,  "lr": 1e-3, "weight_decay": 1e-4},
        {"stage": 2, "name": "+ Layer 4",           "epochs": 5,  "lr": 1e-4, "weight_decay": 1e-4},
        {"stage": 3, "name": "+ Layer 3",           "epochs": 5,  "lr": 5e-5, "weight_decay": 1e-4},
        {"stage": 4, "name": "Full Fine-Tuning",    "epochs": 15, "lr": 1e-5, "weight_decay": 1e-4},
    ]
}

CLASS_NAMES = ["AKIEC", "BCC", "BEN_OTH", "BKL", "DF", "INF", "MAL_OTH", "MEL", "NV", "SCCKA", "VASC"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def get_generator(seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return g

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

## 🛡️ Cell 3: Data Loading & Split Synchronization
To ensure 100% scientific validity, we load the **exact same split** from `outputs/base_resnet50/split/` if already generated, or create the identical Seed 42 lesion-level split.

In [ ]:
def locate_files(data_dir):
    root = Path(data_dir)
    meta_files = [p for p in root.rglob("*.csv") if "metadata" in p.name.lower() and "train" in p.name.lower()]
    if not meta_files:
        meta_files = [p for p in root.rglob("*metadata*.csv") if "test" not in p.name.lower() and "isic_2019" not in p.name.lower()]
    if not meta_files: meta_files = [p for p in root.rglob("*metadata*.csv")]
    meta_file = meta_files[0]

    gt_files = [p for p in root.rglob("*.csv") if "groundtruth" in p.name.lower() and "train" in p.name.lower()]
    if not gt_files: gt_files = [p for p in root.rglob("*groundtruth*.csv")]
    gt_file = gt_files[0]

    img_dirs = [p for p in root.rglob("*") if p.is_dir() and "input" in p.name.lower() and "train" in p.name.lower()]
    if not img_dirs: img_dirs = [p for p in root.rglob("*") if p.is_dir() and "input" in p.name.lower()]
    if img_dirs:
        leaf_dirs = [d for d in img_dirs if any(d.glob("*.jpg")) or any(d.glob("*/*.jpg")) or any(d.glob("*.png"))]
        img_dir = leaf_dirs[-1] if leaf_dirs else img_dirs[0]
    else:
        jpg_sample = list(root.rglob("*.jpg")) + list(root.rglob("*.png"))
        img_dir = jpg_sample[0].parent.parent if jpg_sample and jpg_sample[0].parent != root else root
    return meta_file, gt_file, img_dir

meta_file, gt_file, img_dir = locate_files(CONFIG["data_dir"])
meta_df = pd.read_csv(meta_file)
gt_df   = pd.read_csv(gt_file)
gt_df["class_index"] = gt_df[CLASS_NAMES].values.argmax(axis=1)
gt_df["diagnosis"]   = gt_df["class_index"].map(lambda i: CLASS_NAMES[i])

# Build image path index
valid_exts = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
path_index = {}
for p in img_dir.rglob("*"):
    if p.suffix in valid_exts and p.is_file():
        path_index[p.stem] = str(p.resolve())
if len(path_index) < 10480:
    for p in Path(CONFIG["data_dir"]).rglob("*"):
        if p.suffix in valid_exts and p.is_file():
            path_index[p.stem] = str(p.resolve())

merged = meta_df.merge(gt_df[["lesion_id", "diagnosis", "class_index"]], on="lesion_id", how="inner")
merged["image_id"] = merged["isic_id"]
merged["image_path"] = merged["image_id"].map(path_index)

# Check if base split exists; if so, re-use identical split
base_split_file = Path("outputs/base_resnet50/split/lesion_split.csv")
if base_split_file.exists():
    print(f"✓ Loading exact split file from '{base_split_file}'...")
    split_df = pd.read_csv(base_split_file)
    split_map = dict(zip(split_df["lesion_id"], split_df["split"]))
else:
    print("Creating identical lesion-level stratified split (Seed 42)...")
    lesion_table = merged[["lesion_id", "class_index"]].drop_duplicates("lesion_id").reset_index(drop=True)
    tr_val, te = train_test_split(lesion_table, test_size=0.15, random_state=42, stratify=lesion_table["class_index"])
    tr, va     = train_test_split(tr_val, test_size=0.15/0.85, random_state=42, stratify=tr_val["class_index"])
    tr["split"], va["split"], te["split"] = "train", "val", "test"
    split_df = pd.concat([tr, va, te], ignore_index=True)
    split_map = dict(zip(split_df["lesion_id"], split_df["split"]))

merged["split"] = merged["lesion_id"].map(split_map)
# Dermoscopic only
train_derm = merged[(merged["split"] == "train") & (merged["image_type"].str.lower().str.contains("dermoscopic"))].copy().reset_index(drop=True)
val_derm   = merged[(merged["split"] == "val") & (merged["image_type"].str.lower().str.contains("dermoscopic"))].copy().reset_index(drop=True)
test_derm  = merged[(merged["split"] == "test") & (merged["image_type"].str.lower().str.contains("dermoscopic"))].copy().reset_index(drop=True)

print(f"✓ Dataset Subsets (Dermoscopic only):")
print(f"  Train : {len(train_derm):,} images / lesions")
print(f"  Val   : {len(val_derm):,} images / lesions")
print(f"  Test  : {len(test_derm):,} images / lesions")

## 🔄 Cell 4: Transforms & DataLoaders

In [ ]:
class MILK10KDataset(Dataset):
    def __init__(self, df, transform=None):
        self.image_paths = df["image_path"].tolist()
        self.labels = df["class_index"].tolist()
        self.lesion_ids = df["lesion_id"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx], self.lesion_ids[idx]

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

gen = get_generator(CONFIG["seed"])
train_loader = DataLoader(
    MILK10KDataset(train_derm, transform=train_transform),
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker,
    generator=gen
)
val_loader = DataLoader(
    MILK10KDataset(val_derm, transform=eval_transform),
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker
)
test_loader = DataLoader(
    MILK10KDataset(test_derm, transform=eval_transform),
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available(),
    worker_init_fn=seed_worker
)
print("✓ DataLoaders ready.")

## 🧊 Cell 5: Gradual Unfreezing Controller
Implements the exact logic to freeze and unfreeze blocks progressively:
- **Stage 1 (Head Only)**: Freeze all parameters, thaw only `model.fc`.
- **Stage 2 (+ layer4)**: Thaw `model.layer4` and `model.fc`.
- **Stage 3 (+ layer3)**: Thaw `model.layer3`, `model.layer4`, and `model.fc`.
- **Stage 4 (Full Fine-Tuning)**: Thaw all parameters.

In [ ]:
def create_resnet50_model(num_classes=11):
    model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def set_unfreezing_stage(model: nn.Module, stage_num: int) -> None:
    """
    Sets the active trainable parameters based on stage number:
      Stage 1: Head Only (FC)
      Stage 2: Head + Layer4
      Stage 3: Head + Layer4 + Layer3
      Stage 4: Full network fine-tuning
    """
    # 1. Freeze everything by default
    for param in model.parameters():
        param.requires_grad = False

    if stage_num == 1:
        # Head Only
        for param in model.fc.parameters():
            param.requires_grad = True
        desc = "Stage 1: Head Only [model.fc]"

    elif stage_num == 2:
        # + layer4
        for param in model.layer4.parameters():
            param.requires_grad = True
        for param in model.fc.parameters():
            param.requires_grad = True
        desc = "Stage 2: + Layer 4 [model.layer4, model.fc]"

    elif stage_num == 3:
        # + layer3
        for param in model.layer3.parameters():
            param.requires_grad = True
        for param in model.layer4.parameters():
            param.requires_grad = True
        for param in model.fc.parameters():
            param.requires_grad = True
        desc = "Stage 3: + Layer 3 [model.layer3, model.layer4, model.fc]"

    elif stage_num >= 4:
        # Full fine-tuning
        for param in model.parameters():
            param.requires_grad = True
        desc = "Stage 4: Full Network Fine-Tuning [All Layers Trainable]"

    total_p = sum(p.numel() for p in model.parameters())
    train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    pct = (train_p / total_p) * 100
    print(f"\n>>> {desc}")
    print(f"    Trainable Parameters: {train_p:,} / {total_p:,} ({pct:.2f}%)")

## 🏃 Cell 6: Step Engines & Metrics

In [ ]:
def compute_metrics(y_true, y_pred, loss_val):
    return {
        "loss": float(loss_val),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    }

def train_one_epoch(model, loader, criterion, optimizer, device, scaler, use_amp):
    model.train()
    running_loss = 0.0
    preds, targets = [], []
    for imgs, tgts, _ in loader:
        imgs, tgts = imgs.to(device, non_blocking=True), tgts.to(device, non_blocking=True)
        optimizer.zero_grad()
        if use_amp and scaler:
            with torch.cuda.amp.autocast():
                outputs = model(imgs)
                loss = criterion(outputs, tgts)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(imgs)
            loss = criterion(outputs, tgts)
            loss.backward()
            optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        preds.extend(outputs.argmax(dim=1).detach().cpu().numpy())
        targets.extend(tgts.cpu().numpy())
    return running_loss / len(loader.dataset), np.array(targets), np.array(preds)

def evaluate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    preds, targets, lesions = [], [], []
    with torch.no_grad():
        for imgs, tgts, les in loader:
            imgs, tgts = imgs.to(device, non_blocking=True), tgts.to(device, non_blocking=True)
            outputs = model(imgs)
            loss = criterion(outputs, tgts)
            running_loss += loss.item() * imgs.size(0)
            preds.extend(outputs.argmax(dim=1).detach().cpu().numpy())
            targets.extend(tgts.cpu().numpy())
            lesions.extend(les)
    return running_loss / len(loader.dataset), np.array(targets), np.array(preds), lesions

## 🚀 Cell 7: Progressive 4-Stage Execution Loop
Trains the model across Stage 1, Stage 2, Stage 3, and Stage 4 sequentially.
Continuously monitors **Validation Macro F1** and preserves the overall best checkpoint.

In [ ]:
out_dir = Path(CONFIG["output_dir"])
ckpt_dir = out_dir / "checkpoints"
logs_dir = out_dir / "logs"
ckpt_dir.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

best_ckpt_path = ckpt_dir / "best_gradual_unfreezing_resnet50.pth"
last_ckpt_path = ckpt_dir / "last_gradual_unfreezing_resnet50.pth"

model = create_resnet50_model(num_classes=len(CLASS_NAMES)).to(device)
criterion = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler() if CONFIG["use_amp"] and torch.cuda.is_available() else None

best_val_macro_f1 = -1.0
best_epoch_global = 0
global_epoch = 0
all_history = []

print("=" * 75)
print("STARTING GRADUAL UNFREEZING RESNET50 (4 STAGES)")
print("=" * 75)

start_time = time.time()

for stage_info in CONFIG["stages"]:
    stage_num = stage_info["stage"]
    stage_name = stage_info["name"]
    stage_epochs = stage_info["epochs"]
    stage_lr = stage_info["lr"]
    stage_wd = stage_info["weight_decay"]

    print("\n" + "#" * 75)
    print(f"STAGE {stage_num}/4: {stage_name} | Epochs: {stage_epochs} | LR: {stage_lr}")
    print("#" * 75)

    # 1. Update unfreezing state
    set_unfreezing_stage(model, stage_num)

    # 2. Re-initialize optimizer & scheduler for the newly active trainable parameters
    active_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(active_params, lr=stage_lr, weight_decay=stage_wd)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=stage_epochs)

    # 3. Train epochs for this stage
    for ep in range(1, stage_epochs + 1):
        global_epoch += 1
        t0 = time.time()

        tr_loss, tr_y, tr_pred = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler, CONFIG["use_amp"])
        tr_m = compute_metrics(tr_y, tr_pred, tr_loss)

        va_loss, va_y, va_pred, _ = evaluate_one_epoch(model, val_loader, criterion, device)
        va_m = compute_metrics(va_y, va_pred, va_loss)

        scheduler.step()

        is_best = False
        if va_m["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = va_m["macro_f1"]
            best_epoch_global = global_epoch
            is_best = True
            torch.save({
                "global_epoch": global_epoch,
                "stage": stage_num,
                "stage_name": stage_name,
                "model_state_dict": model.state_dict(),
                "val_metrics": va_m
            }, best_ckpt_path)

        # Save last checkpoint
        torch.save({
            "global_epoch": global_epoch,
            "stage": stage_num,
            "model_state_dict": model.state_dict(),
            "val_metrics": va_m
        }, last_ckpt_path)

        rec = {
            "global_epoch": global_epoch,
            "stage": stage_num,
            "stage_name": stage_name,
            "train_loss": tr_m["loss"],
            "train_accuracy": tr_m["accuracy"],
            "train_balanced_accuracy": tr_m["balanced_accuracy"],
            "train_macro_f1": tr_m["macro_f1"],
            "val_loss": va_m["loss"],
            "val_accuracy": va_m["accuracy"],
            "val_balanced_accuracy": va_m["balanced_accuracy"],
            "val_macro_f1": va_m["macro_f1"],
        }
        all_history.append(rec)

        star = " [*] GLOBAL BEST" if is_best else ""
        print(f"Global Ep [{global_epoch:02d}/30] Stage {stage_num} [{ep:02d}/{stage_epochs:02d}] ({time.time()-t0:.1f}s) | TrLoss: {tr_m['loss']:.4f} Acc: {tr_m['accuracy']:.4f} MF1: {tr_m['macro_f1']:.4f} | ValLoss: {va_m['loss']:.4f} Acc: {va_m['accuracy']:.4f} BAcc: {va_m['balanced_accuracy']:.4f} MF1: {va_m['macro_f1']:.4f}{star}")

history_df = pd.DataFrame(all_history)
history_df.to_csv(logs_dir / "gradual_unfreezing_history.csv", index=False)
print(f"\n✓ Gradual Unfreezing Complete in {(time.time()-start_time)/60:.2f} mins!")
print(f"✓ Overall Best Validation Macro F1: {best_val_macro_f1:.4f} at Global Epoch {best_epoch_global}")

## 🎯 Cell 8: Unseen Test Evaluation (Best Checkpoint)
Loads the best checkpoint and performs final evaluation on the unseen test set.
Calculates bootstrap 95% confidence intervals at the lesion level.

In [ ]:
def compute_lesion_bootstrap_ci(lesion_ids, y_true, y_pred, n_bootstraps=1000, seed=42):
    df_eval = pd.DataFrame({"lesion_id": lesion_ids, "target": y_true, "pred": y_pred})
    unique_lesions = df_eval["lesion_id"].unique()
    n_lesions = len(unique_lesions)
    rng = np.random.RandomState(seed)
    groups = df_eval.groupby("lesion_id")

    accs, bal_accs, f1s = [], [], []
    for _ in range(n_bootstraps):
        sampled = rng.choice(unique_lesions, size=n_lesions, replace=True)
        sampled_df = pd.concat([groups.get_group(lid) for lid in sampled], ignore_index=True)
        t, p = sampled_df["target"].values, sampled_df["pred"].values
        accs.append(accuracy_score(t, p))
        bal_accs.append(balanced_accuracy_score(t, p))
        f1s.append(f1_score(t, p, average="macro", zero_division=0))
    return {
        "acc_ci": (float(np.percentile(accs, 2.5)), float(np.percentile(accs, 97.5))),
        "bal_acc_ci": (float(np.percentile(bal_accs, 2.5)), float(np.percentile(bal_accs, 97.5))),
        "macro_f1_ci": (float(np.percentile(f1s, 2.5)), float(np.percentile(f1s, 97.5)))
    }

eval_model = create_resnet50_model(num_classes=len(CLASS_NAMES)).to(device)
ckpt = torch.load(best_ckpt_path, map_location=device)
eval_model.load_state_dict(ckpt["model_state_dict"])

t_loss, y_true, y_pred, lesion_ids = evaluate_one_epoch(eval_model, test_loader, nn.CrossEntropyLoss(), device)

test_acc = accuracy_score(y_true, y_pred)
test_bacc = balanced_accuracy_score(y_true, y_pred)
test_mprec = precision_score(y_true, y_pred, average="macro", zero_division=0)
test_mrec = recall_score(y_true, y_pred, average="macro", zero_division=0)
test_mf1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
test_wf1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

ci = compute_lesion_bootstrap_ci(lesion_ids, y_true, y_pred, n_bootstraps=1000, seed=42)
rep_dict = classification_report(y_true, y_pred, labels=list(range(len(CLASS_NAMES))), target_names=CLASS_NAMES, output_dict=True, zero_division=0)

rep_rows = []
for c in CLASS_NAMES:
    rep_rows.append({
        "class_name": c,
        "precision": rep_dict[c]["precision"],
        "recall": rep_dict[c]["recall"],
        "f1-score": rep_dict[c]["f1-score"],
        "support": int(rep_dict[c]["support"])
    })
rep_rows.append({"class_name": "macro avg", "precision": test_mprec, "recall": test_mrec, "f1-score": test_mf1, "support": len(y_true)})
rep_rows.append({"class_name": "weighted avg", "precision": rep_dict["weighted avg"]["precision"], "recall": rep_dict["weighted avg"]["recall"], "f1-score": test_wf1, "support": len(y_true)})
rep_df = pd.DataFrame(rep_rows)

res_dir = out_dir / "results"
res_dir.mkdir(parents=True, exist_ok=True)
rep_df.to_csv(res_dir / "classification_report.csv", index=False)

cm_raw = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
with np.errstate(all='ignore'):
    cm_norm = np.nan_to_num(cm_raw.astype("float") / cm_raw.sum(axis=1)[:, np.newaxis])

gradual_results = {
    "accuracy": float(test_acc),
    "balanced_accuracy": float(test_bacc),
    "macro_precision": float(test_mprec),
    "macro_recall": float(test_mrec),
    "macro_f1": float(test_mf1),
    "weighted_f1": float(test_wf1),
    "accuracy_95_ci": ci["acc_ci"],
    "balanced_accuracy_95_ci": ci["bal_acc_ci"],
    "macro_f1_95_ci": ci["macro_f1_ci"],
    "mal_oth_metrics": {
        "precision": rep_dict["MAL_OTH"]["precision"],
        "recall": rep_dict["MAL_OTH"]["recall"],
        "f1": rep_dict["MAL_OTH"]["f1-score"],
        "support": int(rep_dict["MAL_OTH"]["support"])
    }
}
with open(res_dir / "test_metrics.json", "w") as f:
    json.dump(gradual_results, f, indent=2)

print("=" * 75)
print("GRADUAL UNFREEZING RESNET50: FINAL TEST EVALUATION")
print("=" * 75)
print(f"Test Accuracy          : {test_acc:.4f} (95% CI: [{ci['acc_ci'][0]:.4f}, {ci['acc_ci'][1]:.4f}])")
print(f"Test Balanced Accuracy : {test_bacc:.4f} (95% CI: [{ci['bal_acc_ci'][0]:.4f}, {ci['bal_acc_ci'][1]:.4f}])")
print(f"Test Macro Precision   : {test_mprec:.4f}")
print(f"Test Macro Recall      : {test_mrec:.4f}")
print(f"Test Macro F1          : {test_mf1:.4f} (95% CI: [{ci['macro_f1_ci'][0]:.4f}, {ci['macro_f1_ci'][1]:.4f}])")
print(f"Test Weighted F1       : {test_wf1:.4f}")
print("\nFull Per-Class Classification Report:")
print(rep_df.to_string(index=False))

## 📊 Cell 9: Stage Transition Curves & Confusion Matrices

In [ ]:
fig_dir = out_dir / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

# 1. Progressive Stage Curves
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot Loss with stage vertical lines
axes[0].plot(history_df["global_epoch"], history_df["train_loss"], label="Train Loss", lw=2, color="#1f77b4")
axes[0].plot(history_df["global_epoch"], history_df["val_loss"], label="Val Loss", lw=2, color="#ff7f0e", linestyle="--")
axes[0].axvline(5.5, color="grey", linestyle=":", label="Stage Boundaries")
axes[0].axvline(10.5, color="grey", linestyle=":")
axes[0].axvline(15.5, color="grey", linestyle=":")
axes[0].set_title("4-Stage Loss Curve", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Global Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend()

# Plot Macro F1 with stage vertical lines
axes[1].plot(history_df["global_epoch"], history_df["train_macro_f1"], label="Train Macro F1", lw=2, color="#17becf")
axes[1].plot(history_df["global_epoch"], history_df["val_macro_f1"], label="Val Macro F1", lw=2, color="#e377c2", linestyle="--")
axes[1].axvline(5.5, color="grey", linestyle=":", label="Stage Boundaries")
axes[1].axvline(10.5, color="grey", linestyle=":")
axes[1].axvline(15.5, color="grey", linestyle=":")
axes[1].scatter([best_epoch_global], [best_val_macro_f1], color="red", s=100, zorder=5, label=f"Best Val F1 ({best_val_macro_f1:.4f})")
axes[1].set_title("4-Stage Macro F1 Optimization", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Global Epoch"); axes[1].set_ylabel("Macro F1"); axes[1].legend()

plt.tight_layout()
plt.savefig(fig_dir / "gradual_unfreezing_curves.png", dpi=300)
plt.show()

# 2. Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(cm_raw, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title("Gradual Unfreezing: Raw Confusion Matrix", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True"); axes[0].tick_params(axis='x', rotation=45)

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title("Gradual Unfreezing: Row-Normalized Matrix (Recall)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True"); axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(fig_dir / "confusion_matrices.png", dpi=300)
plt.show()

## ⚖️ Cell 10: Comparative Research Benchmark (Base vs Gradual Unfreezing)
Directly compares the **Base ResNet50** benchmark with this **Gradual Unfreezing** benchmark.

In [ ]:
base_metrics_file = Path(CONFIG["base_model_results_dir"]) / "test_metrics.json"
has_base = base_metrics_file.exists()

if has_base:
    with open(base_metrics_file, "r") as f:
        base_m = json.load(f)
else:
    # Default placeholder from standard Base ResNet50 benchmark
    base_m = {
        "accuracy": 0.7265,
        "balanced_accuracy": 0.4362,
        "macro_f1": 0.4337,
        "weighted_f1": 0.7100
    }

comparison_rows = [
    {
        "Metric": "Test Accuracy",
        "Base ResNet50": f"{base_m['accuracy']:.4f}",
        "Gradual Unfreezing": f"{gradual_results['accuracy']:.4f}",
        "Absolute Gain (Δ)": f"{(gradual_results['accuracy'] - base_m['accuracy']):+.4f}"
    },
    {
        "Metric": "Balanced Accuracy",
        "Base ResNet50": f"{base_m['balanced_accuracy']:.4f}",
        "Gradual Unfreezing": f"{gradual_results['balanced_accuracy']:.4f}",
        "Absolute Gain (Δ)": f"{(gradual_results['balanced_accuracy'] - base_m['balanced_accuracy']):+.4f}"
    },
    {
        "Metric": "Macro F1 (Primary)",
        "Base ResNet50": f"{base_m['macro_f1']:.4f}",
        "Gradual Unfreezing": f"{gradual_results['macro_f1']:.4f}",
        "Absolute Gain (Δ)": f"{(gradual_results['macro_f1'] - base_m['macro_f1']):+.4f}"
    },
    {
        "Metric": "Weighted F1",
        "Base ResNet50": f"{base_m['weighted_f1']:.4f}",
        "Gradual Unfreezing": f"{gradual_results['weighted_f1']:.4f}",
        "Absolute Gain (Δ)": f"{(gradual_results['weighted_f1'] - base_m['weighted_f1']):+.4f}"
    }
]

comp_df = pd.DataFrame(comparison_rows)
comp_df.to_csv(res_dir / "benchmark_comparison_base_vs_gradual.csv", index=False)

print("=" * 75)
print("BENCHMARK COMPARISON: BASE RESNET50 vs. GRADUAL UNFREEZING")
print("=" * 75)
print(comp_df.to_string(index=False))
print("=" * 75)